In [127]:
# https://platform.olimpiada-ai.ro/en/problems/202

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import string

In [128]:
train = pd.read_csv("/kaggle/input/datasets/abukanabek/juditis-mission/train.csv")
test = pd.read_csv("/kaggle/input/datasets/abukanabek/juditis-mission/test.csv")

train.shape, test.shape

((505, 103), (100, 101))

In [129]:
train.head()

,id,name,powerstats__intelligence,powerstats__strength,powerstats__speed,powerstats__durability,powerstats__power,powerstats__combat,Publisher,Gender,...,Super Breath,Wallcrawling,Vision - Night,Vision - Infrared,Vision - Heat,Vision - X-Ray,Vision - Thermal,meta_score,efficiency_delta,stability_index
0,179,Deathlok,67.854503,33.292252,30.244400,70.477368,40.627622,60.0,Marvel Comics,Male,...,False,False,False,True,False,False,False,0.273676,-1.577509,D
1,112,Box IV,49.574068,74.901682,24.369510,28.579236,10.886175,56.0,Marvel Comics,-,...,False,False,False,False,False,False,False,-1.517710,-8.211461,D
2,272,Hercules,64.491540,100.738794,45.815660,82.916665,90.097416,100.0,Marvel Comics,Male,...,False,False,False,False,False,False,False,-0.688044,8.189362,B
3,438,Phantom,-0.775359,NaN,-1.372187,-0.403532,-0.167523,-1.0,DC Comics,Male,...,False,False,False,False,False,False,False,0.138805,-16.458575,D
4,537,Spider-Woman IV,0.065715,0.078526,0.156320,-1.934220,0.859594,-1.0,Marvel Comics,Female,...,False,True,False,False,False,False,False,0.009787,-1.440590,C


In [130]:
def delete_punctuation(text):
    for ch in string.punctuation:
        text = text.replace(ch, ' ')
    return text

def preprocess(df):
    df['name'] = df['name'].map(lambda x: delete_punctuation(x))
    df.loc[:, df.select_dtypes('object').columns] = df.loc[:, df.select_dtypes('object').columns].fillna('-')

preprocess(train)
preprocess(test)

In [131]:
from sklearn.model_selection import train_test_split
from catboost import Pool, CatBoostRegressor, CatBoostClassifier

def fit(target_col):
    features = [c for c in test.columns if c not in ['id', target_col]]
    text_features = ['name']
    cat_features = [c for c in train.select_dtypes('object').columns if c not in text_features]

    X, y = train[features], train[target_col]
    X_test = test[features]

    X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=42, test_size=0.1)
    train_pool = Pool(X_train, y_train, cat_features=cat_features, text_features=text_features)
    valid_pool = Pool(X_valid, y_valid, cat_features=cat_features, text_features=text_features)

    if target_col == 'powerstats__combat':
        model = CatBoostRegressor(
            iterations=200,
            loss_function='MAE',
            eval_metric='MAE',
            metric_period=20,
            max_depth=3,
            random_state=42,
            learning_rate=0.1
        )
    else:
        model = CatBoostClassifier(
            iterations=100,
            loss_function='Logloss',
            eval_metric='Accuracy',
            metric_period=10,
            max_depth=2,
            random_state=42,
            learning_rate=0.1
        )

    model.fit(train_pool, eval_set=valid_pool)

    y_pred = model.predict(X_test)
    
    return model, y_pred

In [132]:
model1, y_pred1 = fit('powerstats__combat')
model2, y_pred2 = fit('Super Strength')

0:	learn: 24.1511003	test: 20.0509794	best: 20.0509794 (0)	total: 2.13ms	remaining: 425ms
20:	learn: 14.7162208	test: 14.6586992	best: 14.6586992 (20)	total: 36.9ms	remaining: 315ms
40:	learn: 13.0916072	test: 13.6931585	best: 13.6931585 (40)	total: 69.5ms	remaining: 270ms
60:	learn: 12.3428371	test: 13.4577520	best: 13.4577520 (60)	total: 103ms	remaining: 234ms
80:	learn: 11.6329483	test: 13.3483800	best: 13.3483800 (80)	total: 136ms	remaining: 200ms
100:	learn: 10.9811335	test: 13.2040305	best: 13.2040305 (100)	total: 171ms	remaining: 167ms
120:	learn: 10.4230187	test: 13.3756655	best: 13.2040305 (100)	total: 207ms	remaining: 135ms
140:	learn: 10.0285037	test: 13.3944796	best: 13.2040305 (100)	total: 241ms	remaining: 101ms
160:	learn: 9.6729877	test: 13.2398011	best: 13.2040305 (100)	total: 277ms	remaining: 67ms
180:	learn: 9.3434683	test: 13.2396044	best: 13.2040305 (100)	total: 311ms	remaining: 32.7ms
199:	learn: 9.1043115	test: 13.1887446	best: 13.1887446 (199)	total: 343ms	remain

In [133]:
subtask1 = train['Publisher'].nunique()
subtask2 = train[train['Alignment']=='good']['Publisher'].value_counts().index[0]

In [134]:
subm = pd.DataFrame({
    'subtaskID': ['task1', 'task2'] + ['task3']*len(test) + ['task4']*len(test),
    'id': ['GLOBAL']*2 + test['id'].tolist()*2,
    'answer': [subtask1, subtask2] + y_pred1.tolist() + y_pred2.tolist()
})

subm.to_csv("submission.csv", index=False)

subm.head()

,subtaskID,id,answer
0,task1,GLOBAL,23
1,task2,GLOBAL,Marvel Comics
2,task3,560,83.430102
3,task3,626,60.953349
4,task3,486,50.59694
